# Eigenvector Centrality

Eigenvector Centrality é uma medida de centralidade que atribui pontuações relativas a todos os nós em uma rede, baseada na ideia de que conexões para nós de alta pontuação contribuem mais para a centralidade do nó do que conexões para nós de baixa pontuação. Em outras palavras, um nó é importante se ele está conectado a outros nós importantes. É frequentemente usado para identificar a influência de um nó em uma rede. Um exemplo clássico é o algoritmo PageRank do Google.

## Relação com o Conceito de Autovetor
Matematicamente, a Centralidade de Autovetor deriva do cálculo do autovetor principal da matriz de adjacência do grafo. Um autovetor é um vetor especial que, quando transformado por uma matriz (neste caso, a matriz que representa as conexões do grafo), tem sua direção mantida, apenas sua magnitude é escalada por um fator, chamado autovalor. No contexto da centralidade, o autovetor principal (associado ao maior autovalor) fornece um conjunto de valores para cada nó do grafo. Esses valores são as pontuações de centralidade, indicando que um nó é central se seus vizinhos também são centrais. Quanto maior o valor no autovetor para um determinado nó, maior sua centralidade de autovetor e, consequentemente, maior sua influência dentro da rede.

## Aplicação em Auditorias de Contratações Públicas

Em auditorias de contratações públicas, o Eigenvector Centrality pode ser usado para identificar entidades (empresas, pessoas, órgãos) que, embora talvez não estejam no centro de muitas transações diretas (alta Betweenness Centrality), exercem grande influência por estarem conectadas a outras entidades influentes. Isso é crucial para detectar redes de empresas que operam de forma coordenada, esquemas de direcionamento de licitações ou indivíduos que, nos bastidores, controlam múltiplas empresas ou têm conexões-chave com agentes públicos. Ao analisar a centralidade de autovetor, auditores podem desvendar estruturas de poder ocultas e focar investigações em pontos estratégicos da rede de contratações.

## Sobre este notebook

Usaremos a sua conta gratuita no Neo4J Aura em: https://neo4j.com/cloud/platform/auradb criada no notebook anterior

Copie os dados de conexão utilizados anteriormente na célula abaixo:

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env')

NEO4J_URI=os.getenv("NEO4J_URI")
NEO4J_USERNAME=os.getenv("NEO4J_USERNAME")
NEO4J_DATABASE=os.getenv("NEO4J_DATABASE")
AURA_INSTANCEID=os.getenv("AURA_INSTANCEID")
AURA_INSTANCENAME=os.getenv("AURA_INSTANCENAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

### 1. Instalação do Driver Python para Neo4j

Para interagir com o banco de dados Neo4j a partir do Python, precisamos instalar a biblioteca `neo4j`. O comando `!pip install neo4j` faz essa instalação no ambiente do Colab.

In [2]:
!pip install neo4j

Defaulting to user installation because normal site-packages is not writeable


### 2. Importação de Bibliotecas

Esta célula realiza a importação das bibliotecas necessárias (`pandas` para manipulação de dados, `GraphDatabase` do `neo4j` para conexão e `google.colab.drive` para acesso ao Drive).

In [3]:
import pandas as pd
from neo4j import GraphDatabase
#from google.colab import drive
import os


Testando a instância

### 3. Teste de Conexão

Este bloco de código estabelece uma conexão inicial com o banco de dados Neo4j usando as credenciais fornecidas. A linha `driver.verify_connectivity()` confirma se a conexão foi bem-sucedida.

Verifique no Neo4J Aura Bloom o estado do seu grafo

In [4]:
with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME,NEO4J_PASSWORD)) as driver:

    driver.verify_connectivity()



### 4. Criação da Classe `Neo4jConn` para Interação com o Banco de Dados

Para facilitar a interação com o Neo4j, criamos uma classe `Neo4jConn`. Ela encapsula a lógica de conexão (`__init__`) e o método `query` para executar comandos Cypher. Isso torna o código mais organizado e reutilizável. O teste de conexão final verifica se a classe consegue se comunicar e obter a versão do Neo4j.

In [5]:
class Neo4jConn:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def query(self, query_str, parameters=None):
        with self.driver.session() as session:
            result = session.run(query_str, parameters or {})
            return [record.data() for record in result]

# Teste de Conexão
db = Neo4jConn(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)
print("Conexão bem-sucedida! Versão do Neo4j:", db.query("CALL dbms.components() YIELD name, versions, edition WHERE name = 'Neo4j Kernel' RETURN versions[0] AS neo4jVersion"))

Conexão bem-sucedida! Versão do Neo4j: [{'neo4jVersion': '5.27-aura'}]


### 5. Criação da Projeção de Grafo no Neo4j GDS (In-Memory Graph)
Para executar algoritmos como o Betweenness Centrality no Neo4j, primeiro projetamos o subgrafo relevante na memória usando o módulo GDS (Graph Data Science).

#### 5.1 Limpar projeção anterior se existir

In [6]:
db.query("CALL gds.graph.drop('grafo-auditoria', false)")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. `schema` returned by the procedure `gds.graph.drop` is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('grafo-auditoria', false)"


[{'graphName': 'grafo-auditoria',
  'database': 'neo4j',
  'databaseLocation': 'none',
  'memoryUsage': '',
  'sizeInBytes': -1,
  'nodeCount': 145,
  'relationshipCount': 319,
  'configuration': {'jobId': 'jid-550b5c14-d09f-465c-96b4-2a1383367627',
   'logProgress': True,
   'readConcurrency': 4,
   'sudo': False,
   'validateRelationships': False},
  'density': 0.015277777777777777,
  'creationTime': neo4j.time.DateTime(2026, 8, 21, 17, 46, 23, 475044107, tzinfo=<StaticTzInfo 'GMT'>),
  'modificationTime': neo4j.time.DateTime(2026, 8, 21, 17, 46, 23, 475044107, tzinfo=<StaticTzInfo 'GMT'>),
  'schema': {'relationships': {'PROMOVIDA_POR': {},
    'PARTICIPOU_DE': {},
    'PRESTA_SERVICO_PARA': {}},
   'nodes': {'Licitacao': {}, 'Empresa': {}, 'Orgao': {}, 'Contador': {}}},
  'schemaWithOrientation': {'relationships': {'PROMOVIDA_POR': {'properties': {},
     'direction': 'DIRECTED'},
    'PARTICIPOU_DE': {'properties': {}, 'direction': 'DIRECTED'},
    'PRESTA_SERVICO_PARA': {'propert

#### 5.2 Criar a projeção não-direcionada envolvendo Pessoas, Empresas e Órgãos

**O que é uma Projeção de Grafo?**

Uma projeção de grafo é a criação de um subgrafo em memória (ou seja, uma representação temporária do grafo) a partir de um grafo maior armazenado no banco de dados. Isso é feito para otimizar a execução de algoritmos de grafo (como o Betweenness Centrality).

Ao invés de carregar o grafo inteiro, selecionamos apenas os tipos de nós e relacionamentos relevantes para a análise. Isso economiza memória, melhora a performance e evita que o algoritmo processe dados desnecessários. A projeção `grafo-auditoria` que criamos inclui  `Empresas`, `Orgãos`, `Licitações` e `Contadores`, e os tipos de relacionamentos específicos que nos interessam.

In [7]:
cypher_projection = """
CALL gds.graph.project(
  'grafo-auditoria',
  ['Pessoa', 'Empresa', 'Orgao', 'Licitacao', 'Contador'],
  ['E_SOCIO_DE', 'PRESTA_SERVICO_PARA', 'PROMOVIDA_POR', 'PARTICIPOU_DE'],
  { memory: '2GB' }
)
YIELD graphName, nodeCount, relationshipCount
"""

res_proj = db.query(cypher_projection)
print(f"✓ Projeção '{res_proj[0]['graphName']}' criada com {res_proj[0]['nodeCount']} nós e {res_proj[0]['relationshipCount']} relações!")

✓ Projeção 'grafo-auditoria' criada com 245 nós e 463 relações!


### 6. Executar o Eigenvecto Centrality

In [8]:
cypher_eigenvector = """
CALL gds.eigenvector.stream('grafo-auditoria')
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS node, score
WHERE score > 0 AND NOT 'Orgao' IN labels(node) AND NOT 'Licitacao' IN labels(node)
RETURN
    labels(node)[0] AS TipoEntidade,
    coalesce(node.nome_contador, node.nome, node.razao_social, node.nome_orgao, node.codigo_licitacao) AS Nome,
    coalesce(node.registro_crc, node.cpf, node.cnpj, node.codigo_orgao, node.codigo_licitacao) AS Identificador,
    score AS EigenvectorScore
ORDER BY EigenvectorScore DESC
LIMIT 20
"""

res_eigenvector = db.query(cypher_eigenvector)
df_eigenvector = pd.DataFrame(res_eigenvector)
df_eigenvector.EigenvectorScore = df_eigenvector.EigenvectorScore

print("🏆 Top 15 Entidades com Maior Eigenvector Centrality (excluindo Órgãos e Licitações):")
display(df_eigenvector)

🏆 Top 15 Entidades com Maior Eigenvector Centrality (excluindo Órgãos e Licitações):


,TipoEntidade,Nome,Identificador,EigenvectorScore
0,Empresa,Líder Empreendimentos e Serviços S.A.,09.169.985/0001-58,0.00005
1,Empresa,Vanguard Suprimentos ME,13.389.083/0001-02,0.00005
2,Empresa,Integra Serviços S.A.,48.951.343/0001-68,0.00005
3,Empresa,Inova Alimentos e Serviços Ltda.,60.366.909/0001-90,0.00005
4,Empresa,Visão Logística e Serviços EPP,75.255.341/0001-07,0.00005
5,Empresa,Integra Suprimentos Ltda.,26.247.317/0001-10,0.00005
6,Empresa,Apex Tecnologia e Serviços Eireli,48.281.489/0001-43,0.00005
7,Empresa,Apex Gestão Ltda.,80.132.677/0001-12,0.00005
8,Empresa,Beta Suprimentos e Serviços S.A.,20.163.287/0001-88,0.00005
9,Empresa,Global Auditoria e Serviços Eireli,12.236.231/0001-88,0.00005


### 7. Preparando a visualização

In [9]:
import networkx as nx
import matplotlib.pyplot as plt

# Cypher query to get the central node and its connections up to 2 steps
cypher_graph_data_2_steps = """
MATCH path = (e:Empresa {cnpj: '09.169.985/0001-58'})-[r*1..2]-(target)
UNWIND relationships(path) AS rel
WITH DISTINCT rel
RETURN
    coalesce(startNode(rel).nome_contador, startNode(rel).nome, startNode(rel).razao_social, startNode(rel).nome_orgao, startNode(rel).codigo_licitacao) AS sourceName,
    labels(startNode(rel))[0] AS sourceType,
    type(rel) AS relationshipType,
    coalesce(endNode(rel).nome_contador, endNode(rel).nome, endNode(rel).razao_social, endNode(rel).nome_orgao, endNode(rel).codigo_licitacao) AS targetName,
    labels(endNode(rel))[0] AS targetType,
    coalesce(startNode(rel).cpf, startNode(rel).cnpj, startNode(rel).registro_crc, startNode(rel).codigo_orgao, elementId(startNode(rel))) AS sourceId,
    coalesce(endNode(rel).cpf, endNode(rel).cnpj, endNode(rel).registro_crc, endNode(rel).codigo_orgao, elementId(endNode(rel))) AS targetId
"""

graph_data_2_steps = db.query(cypher_graph_data_2_steps)

# Create a NetworkX graph
G_2_steps = nx.Graph()

# Add nodes and edges
for row in graph_data_2_steps:
    # Add source node
    G_2_steps.add_node(row['sourceId'], label=row['sourceName'], type=row['sourceType'])
    # Add target node
    G_2_steps.add_node(row['targetId'], label=row['targetName'], type=row['targetType'])
    # Add edge
    G_2_steps.add_edge(row['sourceId'], row['targetId'], relationship=row['relationshipType'])

# Set node colors based on type
node_colors_2_steps = []
node_labels_2_steps = {}
# Reuse the existing color_map or define it if not available globally
color_map = {
    'Pessoa': 'skyblue',
    'Empresa': 'lightcoral',
    'Contador': 'lightgreen',
    'Orgao': 'gold',
    'Licitacao': 'plum'
}

for node in G_2_steps.nodes():
    node_type = G_2_steps.nodes[node].get('type', 'unknown')
    node_colors_2_steps.append(color_map.get(node_type, 'gray'))
    node_labels_2_steps[node] = G_2_steps.nodes[node].get('label', node) # Use the 'label' attribute for display

# Set edge labels
edge_labels_2_steps = nx.get_edge_attributes(G_2_steps, 'relationship')



In [10]:
pip install pyvis

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [16]:
from pyvis.network import Network

# Create a Pyvis network object
net = Network(notebook=True, height='750px', width='100%', cdn_resources='remote')

# Add nodes and edges from the NetworkX graph
# Using the G_2_steps graph created previously
for node in G_2_steps.nodes(data=True):
    node_id = node[0]
    node_attrs = node[1]
    net.add_node(node_id, label=node_attrs.get('label', node_id), title=node_attrs.get('label', node_id), group=node_attrs.get('type', 'unknown'))

for edge in G_2_steps.edges(data=True):
    source = edge[0]
    target = edge[1]
    edge_attrs = edge[2]
    net.add_edge(source, target, title=edge_attrs.get('relationship', 'rel'))

# Configure physics and interaction for better usability
net.show_buttons(filter_=['physics', 'nodes', 'edges'])

# Save the network to an HTML file and display it
# You can open the HTML file in a web browser to interact with it
net.save_graph('interactive_graph.html')

In [17]:
from IPython.display import display, HTML

display(HTML(filename='interactive_graph.html'))

## Eigenvector Centrality

Eigenvector Centrality é uma medida de centralidade que atribui pontuações relativas a todos os nós em uma rede, baseada na ideia de que conexões para nós de alta pontuação contribuem mais para a centralidade do nó do que conexões para nós de baixa pontuação. Em outras palavras, um nó é importante se ele está conectado a outros nós importantes. É frequentemente usado para identificar a influência de um nó em uma rede. Um exemplo clássico é o algoritmo PageRank do Google.

### Relação com o Conceito de Autovetor

Matematicamente, a Centralidade de Autovetor deriva do cálculo do autovetor principal da matriz de adjacência do grafo. Um **autovetor** é um vetor especial que, quando transformado por uma matriz (neste caso, a matriz que representa as conexões do grafo), tem sua direção mantida, apenas sua magnitude é escalada por um fator, chamado **autovalor**. No contexto da centralidade, o autovetor principal (associado ao maior autovalor) fornece um conjunto de valores para cada nó do grafo. Esses valores são as pontuações de centralidade, indicando que um nó é central se seus vizinhos também são centrais. Quanto maior o valor no autovetor para um determinado nó, maior sua centralidade de autovetor e, consequentemente, maior sua influência dentro da rede.

### Aplicação em Auditorias de Contratações Públicas

Em auditorias de contratações públicas, o Eigenvector Centrality pode ser usado para identificar entidades (empresas, pessoas, órgãos) que, embora talvez não estejam no centro de muitas transações diretas (alta Betweenness Centrality), exercem grande influência por estarem conectadas a outras entidades influentes. Isso é crucial para detectar redes de empresas que operam de forma coordenada, esquemas de direcionamento de licitações ou indivíduos que, nos bastidores, controlam múltiplas empresas ou têm conexões-chave com agentes públicos. Ao analisar a centralidade de autovetor, auditores podem desvendar estruturas de poder ocultas e focar investigações em pontos estratégicos da rede de contratações.

Cole essa query no Neo4J:
```cypher
MATCH (e:Empresa)-[r*1..2]-(target)
WHERE e.cnpj = '71.331.509/0001-65'
RETURN e, r, target
```